In [ ]:
# ============================================================
# package-alpha: re-empaqueta un adapter con lora_alpha modificado.
# CPU-only (sin GPU, sin internet) — el scale efectivo en vLLM es
# alpha/r, asi que editar alpha re-escala el adapter SIN re-entrenar.
# Editar ALPHA y ADAPTER_DS antes de cada push.
# ============================================================
import json, os, shutil, zipfile, glob

ALPHA = 36            # <-- sweep: 24 / 28 / 36 / 40 (32 = original)
ADAPTER_DS = "gastonz195/v1-best-adapter"

cands = [f"/kaggle/input/datasets/{ADAPTER_DS}/sft_adapter", f"/kaggle/input/datasets/{ADAPTER_DS}"]
src = next(p for p in cands if os.path.exists(os.path.join(p, "adapter_config.json")))
print("adapter src:", src)

OUT = "/kaggle/working/submission_adapter"
os.makedirs(OUT, exist_ok=True)
for f in ("adapter_config.json", "adapter_model.safetensors"):
    shutil.copy2(os.path.join(src, f), os.path.join(OUT, f))

cfg = json.load(open(os.path.join(OUT, "adapter_config.json")))
print("alpha original:", cfg.get("lora_alpha"), "| r:", cfg.get("r"))
cfg["lora_alpha"] = ALPHA
cfg["base_model_name_or_path"] = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
cfg["inference_mode"] = True
json.dump(cfg, open(os.path.join(OUT, "adapter_config.json"), "w"), indent=2)
print("alpha nuevo:", ALPHA, "-> scale efectivo:", ALPHA/cfg["r"])

zp = "/kaggle/working/submission.zip"
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in ("adapter_config.json", "adapter_model.safetensors"):
        zf.write(os.path.join(OUT, f), f)
print("submission.zip:", os.path.getsize(zp)/1e6, "MB")
